# European Power Price Forecasting with Archived Weather

Research only: PriceFM day-ahead prices and Open-Meteo / DWD ICON fixed-lead forecasts. No trading connections or execution. This notebook uses our LightGBM models, not PriceFM neural weights.

Before running: `python data/download_dataset.py` and `python data/download_weather.py --zone DE_LU`. Weather data are CC BY 4.0; the public API has non-commercial terms. [Archive documentation](https://open-meteo.com/en/docs/previous-runs-api).

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'config').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import plotly.express as px
from data.connectors import PriceFMConnector
from data.weather import WeatherConnector, WEATHER_COLUMNS
from features.build_features import build_features
from backtest.walk_forward import walk_forward
from models.gbm import GBM
from models.quantile import QuantileGBM
from models.baselines import Naive
from app.common import forecast_chart
zone = 'DE_LU'
prices = PriceFMConnector(zone).load()
weather = WeatherConnector(zone).load()
print(f'{len(prices):,} price hours; {len(weather):,} weather hours')
print(weather.timestamp.min(), weather.timestamp.max())
weather.head()

2026-09-17 17:38:13.733 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-17 17:38:13.734 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


35,064 price hours; 15,360 weather hours
2024-04-01 00:00:00+00:00 2025-12-31 23:00:00+00:00


,timestamp,reference_time_bound,assumed_available_at,weather_temperature_c,weather_wind_100m_ms,weather_cloud_pct,weather_radiation_wm2,weather_precipitation_mm,weather_temperature_spread_c,weather_heating_degree_c,weather_cooling_degree_c
0,2024-04-01 00:00:00+00:00,2024-03-30 00:00:00+00:00,2024-03-30 08:00:00+00:00,11.225,6.5225,50.0,0.0,0.100,4.1,6.775,0.0
1,2024-04-01 01:00:00+00:00,2024-03-30 01:00:00+00:00,2024-03-30 09:00:00+00:00,10.950,7.0750,70.0,0.0,0.275,4.2,7.050,0.0
2,2024-04-01 02:00:00+00:00,2024-03-30 02:00:00+00:00,2024-03-30 10:00:00+00:00,10.575,7.0975,73.5,0.0,0.150,4.4,7.425,0.0
3,2024-04-01 03:00:00+00:00,2024-03-30 03:00:00+00:00,2024-03-30 11:00:00+00:00,10.175,7.4100,65.5,0.0,0.675,4.5,7.825,0.0
4,2024-04-01 04:00:00+00:00,2024-03-30 04:00:00+00:00,2024-03-30 12:00:00+00:00,9.750,6.6025,75.0,0.0,0.375,4.7,8.250,0.0


## Timing and spatial assumptions
Weather variables use `previous_day2`: a fixed 48-hour offset for each valid hour. This is **not** one common daily forecast run. `reference_time_bound = valid_time - 48h`; `assumed_available_at = reference_time_bound + 8h` must be no later than D−1 11:00 Brussels for each sample. Neither field is an observed publication timestamp. Target-day reanalysis is never substituted.

Representative geographic sites have equal weights, not population/load/generation weights. Instantaneous radiation is aligned to the hour start; precipitation is the preceding-hour accumulation. Heating/cooling thresholds (18/22 C) are illustrative. UTC storage preserves repeated autumn hours; local delivery days have 23/24/25 hours.

In [2]:
meta = WeatherConnector(zone).metadata()
print('Sites:', meta['sites'])
print('Missing hours omitted, without filling:', meta['missing_hours'])
features = build_features(prices, use_exogenous=False, weather=weather)
features[WEATHER_COLUMNS].dropna().describe().round(2)

Sites: [['Hamburg', 53.55, 9.99], ['Berlin', 52.52, 13.41], ['Munich', 48.14, 11.58], ['Luxembourg', 49.61, 6.13]]
Missing hours omitted, without filling: 0


,weather_temperature_c,weather_wind_100m_ms,weather_cloud_pct,weather_radiation_wm2,weather_precipitation_mm,weather_temperature_spread_c,weather_heating_degree_c,weather_cooling_degree_c
count,15359.00,15359.00,15359.00,15359.00,15359.00,15359.00,15359.00,15359.00
mean,11.61,5.85,69.33,137.08,0.09,4.09,7.21,0.26
std,7.28,2.19,25.02,197.62,0.21,2.15,6.13,1.02
min,-6.38,0.72,0.00,0.00,0.00,0.20,0.00,0.00
25%,5.72,4.34,53.75,0.00,0.00,2.50,1.40,0.00
50%,12.00,5.60,75.00,11.22,0.00,3.70,6.02,0.00
75%,17.01,7.06,89.50,228.81,0.08,5.30,12.28,0.00
max,34.70,16.35,100.00,853.50,4.12,16.80,24.38,12.70


In [3]:
aligned = prices.merge(weather, on='timestamp', how='inner')
week = aligned.tail(168)
px.line(week, x='timestamp', y='weather_temperature_c', title='Archived 48-hour-lead temperature forecast (°C)').show()
px.scatter(aligned.iloc[::8], x='weather_wind_100m_ms', y='price', opacity=0.3, title='Regional wind forecast vs price; correlation is not causality').show()

## Paired walk-forward evaluation
Both arms receive the same weather availability mask, including the arm that does not use weather features. Consequently training rows and evaluation dates match. Use the CLI seasonal benchmark for the larger 120-day evaluation. This seven-day walkthrough verifies the workflow; it does not establish stable trading performance. PriceFM load/wind/solar forecasts are disabled here because their original issue vintages are unavailable.

In [4]:
rows, forecasts, diagnostics = [], {}, {}
for name, cls, use_weather in [
    ('Persistence', Naive, False),
    ('LightGBM prices', GBM, False),
    ('LightGBM weather', GBM, True),
    ('Quantile prices', QuantileGBM, False),
    ('Quantile weather', QuantileGBM, True),
]:
    pred, metrics, diag = walk_forward(prices, cls, days=7, end_date='2025-12-31',
        use_exogenous=False, weather=weather, use_weather=use_weather)
    rows.append(metrics.assign(model=name))
    forecasts[name], diagnostics[name] = pred, diag
    print(name, 'MAE:', round(metrics.iloc[0].mae, 3))
table = pd.concat(rows, ignore_index=True)
assert forecasts['LightGBM prices'].timestamp.equals(forecasts['LightGBM weather'].timestamp)
assert [d['train_rows'] for d in diagnostics['LightGBM prices']] == [d['train_rows'] for d in diagnostics['LightGBM weather']]
table[table.delivery_period == 'overall']

Persistence MAE: 10.717


LightGBM prices MAE: 9.296


LightGBM weather MAE: 9.711


Quantile prices MAE: 8.27


Quantile weather MAE: 8.225


,delivery_period,n,mae,rmse,wape,spike_precision,spike_recall,spike_events,spike_predictions,model,pinball_p10,pinball_p50,pinball_p90,coverage_80,interval_width
0,overall,168,10.717307,13.447164,0.120845,NaN,NaN,0,0,Persistence,NaN,NaN,NaN,NaN,NaN
4,overall,168,9.295896,12.841879,0.104817,NaN,NaN,0,0,LightGBM prices,NaN,NaN,NaN,NaN,NaN
8,overall,168,9.711002,13.378540,0.109498,NaN,NaN,0,0,LightGBM weather,NaN,NaN,NaN,NaN,NaN
12,overall,168,8.269556,11.157612,0.093245,NaN,NaN,0,0,Quantile prices,1.583701,4.134778,1.534940,0.821429,26.037465
16,overall,168,8.224805,11.518683,0.092740,NaN,NaN,0,0,Quantile weather,1.772623,4.112402,1.898492,0.761905,24.977327


In [5]:
pred = forecasts['Quantile weather']
forecast_chart(pred).show()
table[table.delivery_period != 'overall']

,delivery_period,n,mae,rmse,wape,spike_precision,spike_recall,spike_events,spike_predictions,model,pinball_p10,pinball_p50,pinball_p90,coverage_80,interval_width
1,weekday_offpeak,60,8.916917,11.401444,0.107991,NaN,NaN,0,0,Persistence,NaN,NaN,NaN,NaN,NaN
2,weekday_peak,60,13.024417,16.090727,0.141467,NaN,NaN,0,0,Persistence,NaN,NaN,NaN,NaN,NaN
3,weekend,48,10.083906,12.114505,0.109482,NaN,NaN,0,0,Persistence,NaN,NaN,NaN,NaN,NaN
5,weekday_offpeak,60,8.607366,10.746366,0.104242,NaN,NaN,0,0,LightGBM prices,NaN,NaN,NaN,NaN,NaN
6,weekday_peak,60,9.096266,11.385384,0.098801,NaN,NaN,0,0,LightGBM prices,NaN,NaN,NaN,NaN,NaN
7,weekend,48,10.406096,16.456286,0.112980,NaN,NaN,0,0,LightGBM prices,NaN,NaN,NaN,NaN,NaN
9,weekday_offpeak,60,9.131990,12.302550,0.110595,NaN,NaN,0,0,LightGBM weather,NaN,NaN,NaN,NaN,NaN
10,weekday_peak,60,7.985455,10.066865,0.086735,NaN,NaN,0,0,LightGBM weather,NaN,NaN,NaN,NaN,NaN
11,weekend,48,12.591701,17.623294,0.136710,NaN,NaN,0,0,LightGBM weather,NaN,NaN,NaN,NaN,NaN
13,weekday_offpeak,60,7.871244,10.019470,0.095327,NaN,NaN,0,0,Quantile prices,1.579001,3.935622,1.150412,0.816667,22.753734


## Quantile quality and risk scenarios
P50 is used for MAE/RMSE. Pinball losses score all three quantiles; coverage near 80% and interval width should be assessed together. Sorting quantiles removes crossings but does not calibrate them. Three quantiles do not determine CVaR or the hourly joint distribution.

In [6]:
from analytics.hedging import price_scenarios, hedge_analysis
last = pred[pred.delivery_date == pred.delivery_date.max()]
past = prices[prices.timestamp < last.timestamp.min()].tail(90 * 24).price
spread = max(past.std(), 10)
low = min(past.min(), last.p10.min()) - 2 * spread
high = max(past.max(), last.p90.max()) + 2 * spread
scenarios = price_scenarios(last[['p10', 'p50', 'p90']], low, high)
risk = hedge_analysis(scenarios, 100, float(past.mean()))
px.line(risk, x='hedge_ratio', y=['expected_loss', 'loss_var', 'loss_cvar'], title='Illustrative fixed-volume buyer risk; assumed tails and dependence').show()

In [7]:
report = ROOT / 'reports/weather/DE_LU_summary.csv'
if report.exists():
    seasonal = pd.read_csv(report)
    display(seasonal[seasonal.delivery_period == 'overall'])
else:
    print('Run python scripts/weather_benchmark.py for four seasonal 30-day comparisons.')

,delivery_period,n,mae,rmse,wape,spike_precision,spike_recall,spike_events,spike_predictions,zone,model,pricefm_forecasts,weather,pinball_p10,pinball_p50,pinball_p90,coverage_80,interval_width
0,overall,2879,21.705940,31.583316,0.254778,0.527027,0.491597,238,222,DE_LU,LightGBM,False,False,NaN,NaN,NaN,NaN,NaN
4,overall,2879,18.426870,26.610031,0.216289,0.601476,0.684874,238,271,DE_LU,LightGBM,False,True,NaN,NaN,NaN,NaN,NaN
8,overall,2879,12.162113,18.926804,0.142755,0.762646,0.823529,238,257,DE_LU,LightGBM,True,False,NaN,NaN,NaN,NaN,NaN
12,overall,2879,11.952524,18.647508,0.140295,0.774319,0.836134,238,257,DE_LU,LightGBM,True,True,NaN,NaN,NaN,NaN,NaN
16,overall,2879,33.110748,48.845619,0.388645,NaN,0.000000,238,0,DE_LU,Persistence,False,False,NaN,NaN,NaN,NaN,NaN
20,overall,2879,21.039673,31.644921,0.246958,0.572368,0.365546,238,152,DE_LU,Quantile LightGBM,False,False,5.790052,10.519836,5.661605,0.564085,41.585525
24,overall,2879,18.117714,27.555176,0.212661,0.689119,0.558824,238,193,DE_LU,Quantile LightGBM,False,True,4.698299,9.058857,4.889157,0.595693,39.017770
28,overall,2879,11.631057,18.744937,0.136522,0.784141,0.747899,238,227,DE_LU,Quantile LightGBM,True,False,2.811703,5.815529,3.120777,0.616186,28.162192
32,overall,2879,11.439774,18.650169,0.134277,0.785088,0.752101,238,228,DE_LU,Quantile LightGBM,True,True,2.815723,5.719887,3.091872,0.615839,28.436818
36,overall,2879,29.123382,42.366879,0.341842,NaN,0.000000,238,0,DE_LU,SARIMAX,False,False,NaN,NaN,NaN,NaN,NaN


## Limits
No publication-vintage audit is possible from these input files alone. Weather timing uses a conservative documented allowance; PriceFM also contains upstream interpolation. Geographic sites are illustrative and forecasts are deliberately older than the latest D−1 run. Weather can help or hurt on unseen periods. These are forecast metrics, not a P&L backtest, and no live trades are performed.